# Business Entity Resolution — end-to-end pipeline

For every Source 1 (reference) entity, find all matching Source 2 / Source 3 records.

**Pipeline**
1. **Normalisation** — accent folding, legal-suffix removal, alias (DBA / formerly / a.k.a.) splitting,
   domain-name handling, address abbreviation & state canonicalisation, optional transliteration of Indic-script names.
2. **Blocking / candidate generation** — per country (open set, no hard-coding), hashed TF-IDF over word uni+bi-grams,
   exact top-K cosine search (`sparse_dot_topn`) on three channels: *name*, *address*, *name+address*. The union is the candidate set.
3. **Features** — RapidFuzz similarities (ratio / token-set / token-sort / partial / Jaro-Winkler) on names, aliases,
   space-less names, transliterations, addresses and house numbers; TF-IDF cosines; *context* features (rank of the pair
   among the S1 entity's candidates and among the candidate record's competing S1 entities).
4. **Matcher** — LightGBM binary classifier (MIT licence) trained on the training ground truth.
5. **Decision** — probability threshold + one-to-one constraint (each S2/S3 record is linked to at most one S1 entity,
   which holds in the training ground truth) tuned directly for macro **F0.5** on a held-out split of Source 1 entities.
6. **Output** — `output/matching_results.tsv`, `output/candidate_pairs.tsv`, then the official validator.

Run this notebook from the `student_resource/` directory. Hardware used: 16 GB RAM, 12 CPU threads (no GPU needed).

In [1]:
# !pip install lightgbm sparse_dot_topn rapidfuzz indic_transliteration pyarrow joblib scikit-learn

In [2]:
import os, gc, re, sys, csv, json, time, unicodedata, subprocess
import numpy as np
import pandas as pd
import scipy.sparse as sp
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.preprocessing import normalize
from sparse_dot_topn import sp_matmul_topn
from rapidfuzz import fuzz
from rapidfuzz.process import cpdist
from rapidfuzz.distance import JaroWinkler
import lightgbm as lgb

DATA_DIR = "dataset"
OUT_DIR = "output"
MODEL_DIR = "models"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

NROWS = None                 # e.g. 100_000 for a quick smoke test (outputs will then fail validation)
N_JOBS = os.cpu_count()
TOPK = {"name": 10, "addr": 10, "comb": 15}   # top-K neighbours per blocking channel
MAX_DF = 20_000              # tokens more frequent than this (per country) are ignored for blocking only
HASH_FEATURES = 2 ** 21
N_VAL_S1 = 100_000           # held-out Source 1 entities for threshold tuning / evaluation
N_TRN_S1 = 300_000           # Source 1 entities whose candidate pairs train the matcher
PAIR_CHUNK = 1_000_000
SEED = 42

T0 = time.time()
def log(msg):
    print(f"[{time.time() - T0:7.0f}s] {msg}", flush=True)

## 1. Normalisation

In [3]:
NULL_TOKENS = {"null", "n/a", "na", "none", "nil", "nan"}

ADDR_MAP = {
    "street": "st", "str": "st", "strt": "st", "road": "rd", "avenue": "ave", "av": "ave",
    "drive": "dr", "lane": "ln", "court": "ct", "place": "pl", "boulevard": "blvd", "bd": "blvd",
    "trail": "trl", "circle": "cir", "highway": "hwy", "parkway": "pkwy", "terrace": "ter",
    "square": "sq", "north": "n", "south": "s", "east": "e", "west": "w",
    "apartment": "unit", "apt": "unit", "suite": "unit", "ste": "unit", "flat": "unit",
    "room": "unit", "rm": "unit", "shop": "unit",
    "number": "no", "num": "no", "nr": "near", "opp": "opposite", "floor": "fl", "flr": "fl",
    "first": "1st", "second": "2nd", "third": "3rd", "building": "bldg", "sector": "sec",
    "district": "dist", "dt": "dist", "taluk": "tq", "taluka": "tq",
    "r": "rue", "chemin": "ch", "impasse": "imp", "allee": "all", "route": "rte",
    "faubourg": "fbg", "saint": "st", "sainte": "st",
}

STATE_CODES = {
    "al": "alabama", "ak": "alaska", "az": "arizona", "ar": "arkansas", "ca": "california",
    "co": "colorado", "ct": "connecticut", "de": "delaware", "fl": "florida", "ga": "georgia",
    "hi": "hawaii", "id": "idaho", "il": "illinois", "in": "indiana", "ia": "iowa",
    "ks": "kansas", "ky": "kentucky", "la": "louisiana", "me": "maine", "md": "maryland",
    "ma": "massachusetts", "mi": "michigan", "mn": "minnesota", "ms": "mississippi",
    "mo": "missouri", "mt": "montana", "ne": "nebraska", "nv": "nevada", "nh": "new hampshire",
    "nj": "new jersey", "nm": "new mexico", "ny": "new york", "nc": "north carolina",
    "nd": "north dakota", "oh": "ohio", "ok": "oklahoma", "or": "oregon", "pa": "pennsylvania",
    "ri": "rhode island", "sc": "south carolina", "sd": "south dakota", "tn": "tennessee",
    "tx": "texas", "ut": "utah", "vt": "vermont", "va": "virginia", "wa": "washington",
    "wv": "west virginia", "wi": "wisconsin", "wy": "wyoming", "dc": "district of columbia",
    "ap": "andhra pradesh", "as": "assam", "br": "bihar", "cg": "chhattisgarh", "dl": "delhi",
    "gj": "gujarat", "hr": "haryana", "hp": "himachal pradesh", "jk": "jammu and kashmir",
    "jh": "jharkhand", "ka": "karnataka", "kl": "kerala", "mp": "madhya pradesh",
    "mh": "maharashtra", "od": "odisha", "pb": "punjab", "rj": "rajasthan", "tnd": "tamil nadu",
    "tg": "telangana", "up": "uttar pradesh", "uk": "uttarakhand", "wb": "west bengal",
    "chd": "chandigarh", "py": "puducherry", "goa": "goa",
}
STATE_FULL = {v: k for k, v in STATE_CODES.items()}
STATE_FULL.update({"keralam": "kl", "orissa": "od", "pondicherry": "py", "telengana": "tg",
                   "uttaranchal": "uk"})
STATE_RE = re.compile(r"\b(" + "|".join(sorted(map(re.escape, STATE_FULL), key=len, reverse=True)) + r")\b")

LEGAL = {
    "inc", "incorporated", "llc", "corp", "corporation", "co", "company", "ltd", "limited",
    "pvt", "private", "plc", "pllc", "pc", "lp", "llp", "lllp", "the", "public", "opc",
    "sarl", "sas", "sasu", "sa", "eurl", "sci", "snc", "gmbh", "ag", "srl",
    "and", "of", "et", "de", "la", "le", "les", "du", "des",
}

ALIAS_RE = re.compile(
    r"\b(?:d\s*/\s*b\s*/\s*a|dba|a\s*/\s*k\s*/\s*a|aka|f\s*/\s*k\s*/\s*a|fka|formerly|t\s*/\s*a|"
    r"trading as|doing business as)\b[:\s]*")
NON_ALNUM = re.compile(r"[^0-9a-z]+")
DOMAIN_RE = re.compile(r"(?:www\.)?([a-z0-9\-]+)\.(?:com|net|org|in|co|fr|biz|info|us|io)\b")
BRACKET_ID = re.compile(r"\(\s*id\s*:?\s*\d+\s*\)")

try:
    from indic_transliteration import sanscript
    from indic_transliteration.sanscript import transliterate
    _INDIC_BLOCKS = ["DEVANAGARI", "BENGALI", "GURMUKHI", "GUJARATI", "ORIYA",
                     "TAMIL", "TELUGU", "KANNADA", "MALAYALAM"]
    INDIC = [getattr(sanscript, b, None) for b in _INDIC_BLOCKS]
except ImportError:
    INDIC = None


def ascii_fold(s):
    s = unicodedata.normalize("NFKD", s)
    return "".join(c for c in s if not unicodedata.combining(c)).encode("ascii", "ignore").decode()


def translit(s):
    """Romanise a name written in an Indic script (Unicode U+0900..U+0D7F); '' otherwise."""
    if INDIC is None:
        return ""
    for ch in s:
        o = ord(ch)
        if 0x0900 <= o < 0x0D80:
            scheme = INDIC[(o - 0x0900) // 0x80]
            if scheme is None:
                return ""
            try:
                return transliterate(s, scheme, sanscript.ITRANS)
            except Exception:
                return ""
    return ""


def norm_name(s):
    """-> (full normalised name, core name without legal words, list of alias parts)."""
    s = ascii_fold(s).lower()
    s = BRACKET_ID.sub(" ", s)
    s = DOMAIN_RE.sub(lambda m: " " + m.group(1).replace("-", "") + " ", s)
    s = s.replace("&", " and ").replace("+", " and ")
    full = " ".join(NON_ALNUM.sub(" ", s).split())
    parts = []
    for p in ALIAS_RE.split(s):
        toks = [t for t in NON_ALNUM.sub(" ", p).split() if t not in LEGAL and t not in NULL_TOKENS]
        if toks:
            parts.append(" ".join(toks))
    core = " ".join(parts) or full
    return full, core, parts


def norm_name_rec(s):
    n = len(s)
    nonascii = sum(ord(c) > 127 for c in s) / n if n else 0.0
    tl = ""
    if nonascii > 0.3:
        t = translit(s)
        if t:
            tl = norm_name(t.lower())[1]
    full, core, parts = norm_name(s)
    has_domain = 1 if DOMAIN_RE.search(s.lower()) else 0
    return (full, core, parts[0] if parts else "", parts[-1] if parts else "", tl,
            nonascii, has_domain)


def norm_addr_rec(s):
    s = ascii_fold(s).lower()
    s = STATE_RE.sub(lambda m: STATE_FULL[m.group(1)], s)
    out = []
    for t in NON_ALNUM.sub(" ", s).split():
        if t in NULL_TOKENS:
            continue
        if t.isdigit():
            t = t.lstrip("0") or "0"
        out.append(ADDR_MAP.get(t, t))
    num_toks = [t for t in out if any(c.isdigit() for c in t)]
    return " ".join(out), " ".join(sorted(set(num_toks))), (num_toks[0] if num_toks else "")


def _norm_batch(names, addrs):
    return [norm_name_rec(x) for x in names], [norm_addr_rec(x) for x in addrs]


NAME_COLS = ["name_full", "name_core", "alias_first", "alias_last", "name_tl", "name_nonascii", "name_domain"]
ADDR_COLS = ["addr", "addr_nums", "addr_hn"]
STR_COLS = ["name_full", "name_core", "alias_first", "alias_last", "name_tl", "addr", "addr_nums", "addr_hn"]


def normalize_frame(df, bs=100_000):
    names = df["business_name"].tolist()
    addrs = df["business_address"].tolist()
    res = Parallel(n_jobs=N_JOBS)(delayed(_norm_batch)(names[i:i + bs], addrs[i:i + bs])
                                  for i in range(0, len(df), bs))
    out = pd.DataFrame({"entity_id": df["entity_id"].to_numpy(dtype=object),
                        "country": df["country"].astype("category").to_numpy()})
    nm = pd.DataFrame([r for b in res for r in b[0]], columns=NAME_COLS)
    ad = pd.DataFrame([r for b in res for r in b[1]], columns=ADDR_COLS)
    del res
    for c in NAME_COLS:
        out[c] = nm[c].values
    for c in ADDR_COLS:
        out[c] = ad[c].values
    for c in STR_COLS:
        out[c] = out[c].astype("string[pyarrow]")
    out["name_nonascii"] = out["name_nonascii"].astype(np.float32)
    out["name_domain"] = out["name_domain"].astype(np.int8)
    return out

# quick sanity check
for x in ["Lumvio Labs DBA: Vantora Home Care Inc", "zeix.com", "Sit & Warf Block  L.L.C. (ID: 20029)",
          "ફર્સ્ટ પ્રોપર્ટીઝ", "Marina Ecole France Sarl"]:
    print(x, "->", norm_name_rec(x))
for x in ["175 Boulevard du PrAcsident Franklin Roosevelt, Bordeaux, Nouvelle-Aquitaine",
          "3405 GLEN AVE, <NULL>, BALTIMORE, MD", "Door No 2418, Ernakulam, Keralam"]:
    print(x, "->", norm_addr_rec(x))

Lumvio Labs DBA: Vantora Home Care Inc -> ('lumvio labs dba vantora home care inc', 'lumvio labs vantora home care', 'lumvio labs', 'vantora home care', '', 0.0, 0)
zeix.com -> ('zeix', 'zeix', 'zeix', 'zeix', '', 0.0, 1)
Sit & Warf Block  L.L.C. (ID: 20029) -> ('sit and warf block l l c', 'sit warf block l l c', 'sit warf block l l c', 'sit warf block l l c', '', 0.0, 0)
ફર્સ્ટ પ્રોપર્ટીઝ -> ('', '', '', '', '', 0.9411764705882353, 0)
Marina Ecole France Sarl -> ('marina ecole france sarl', 'marina ecole france', 'marina ecole france', 'marina ecole france', '', 0.0, 0)
175 Boulevard du PrAcsident Franklin Roosevelt, Bordeaux, Nouvelle-Aquitaine -> ('175 blvd du pracsident franklin roosevelt bordeaux nouvelle aquitaine', '175', '175')
3405 GLEN AVE, <NULL>, BALTIMORE, MD -> ('3405 glen ave baltimore md', '3405', '3405')
Door No 2418, Ernakulam, Keralam -> ('door no 2418 ernakulam kl', '2418', '2418')


## 2. Loading a split

In [4]:
READ_KW = dict(sep="\t", dtype=str, keep_default_na=False, quoting=csv.QUOTE_NONE,
               encoding="utf-8", encoding_errors="replace")


def read_source(split, k):
    return pd.read_csv(f"{DATA_DIR}/{split}/{split}_source{k}.tsv", nrows=NROWS, **READ_KW)


def load_split(split):
    s1 = read_source(split, 1)
    s23 = pd.concat([read_source(split, 2), read_source(split, 3)], ignore_index=True)
    log(f"{split}: read S1={len(s1):,} S2+S3={len(s23):,}")
    S1 = normalize_frame(s1); del s1; gc.collect()
    S23 = normalize_frame(s23); del s23; gc.collect()
    S23["is_s3"] = pd.Series(S23["entity_id"]).str.startswith("S3-").astype(np.int8).values
    log(f"{split}: normalised")
    return S1, S23

## 3. Blocking (candidate generation)

Per country label present in Source 1 (so unseen countries such as France are handled automatically):
hashed binary TF-IDF over word uni+bi-grams, L2-normalised. Very frequent tokens (`MAX_DF`) are masked for the
nearest-neighbour search only. Exact sparse top-K cosine search on three channels; the union of their results
is the candidate set that the matcher scores (this is exactly what goes to `candidate_pairs.tsv`).

In [5]:
HV = HashingVectorizer(n_features=HASH_FEATURES, token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 2),
                       lowercase=False, alternate_sign=False, norm=None, binary=True, dtype=np.float32)


def hash_texts(texts, bs=200_000):
    if len(texts) == 0:
        return sp.csr_matrix((0, HASH_FEATURES), dtype=np.float32)
    parts = Parallel(n_jobs=N_JOBS)(delayed(HV.transform)(texts[i:i + bs]) for i in range(0, len(texts), bs))
    return sp.vstack(parts, format="csr")


def tfidf_pair(ta, tb):
    A, B = hash_texts(ta), hash_texts(tb)
    df_ = np.bincount(A.indices, minlength=HASH_FEATURES) + np.bincount(B.indices, minlength=HASH_FEATURES)
    n = A.shape[0] + B.shape[0]
    idf = (np.log((n + 1) / (df_ + 1)) + 1).astype(np.float32)
    A.data *= idf[A.indices]
    B.data *= idf[B.indices]
    return normalize(A), normalize(B), df_


def mask_frequent(X, keep):
    X = X.copy()
    X.data *= keep[X.indices]
    X.eliminate_zeros()
    return normalize(X)


def topk(A, B, k):
    C = sp_matmul_topn(A, B.T.tocsr(), top_n=k, n_threads=N_JOBS).tocoo()
    return C.row.astype(np.int64), C.col.astype(np.int64)


def pair_dot(A, B, ia, ib, bs=PAIR_CHUNK):
    out = np.empty(len(ia), np.float32)
    for s in range(0, len(ia), bs):
        out[s:s + bs] = np.asarray(A[ia[s:s + bs]].multiply(B[ib[s:s + bs]]).sum(1)).ravel()
    return out


def block_split(S1, S23):
    frames = []
    for c in S1["country"].unique():
        ga = np.flatnonzero((S1["country"] == c).values)
        gb = np.flatnonzero((S23["country"] == c).values)
        if len(gb) == 0:
            log(f"  {c}: no S2/S3 records"); continue
        a1, b1 = S1.iloc[ga], S23.iloc[gb]
        b_name = (b1["name_core"] + " " + b1["name_tl"]).tolist()
        An, Bn, dfn = tfidf_pair(a1["name_core"].tolist(), b_name)
        Aa, Ba, dfa = tfidf_pair(a1["addr"].tolist(), b1["addr"].tolist())
        kn, ka = (dfn <= MAX_DF).astype(np.float32), (dfa <= MAX_DF).astype(np.float32)
        An_m, Bn_m = mask_frequent(An, kn), mask_frequent(Bn, kn)
        Aa_m, Ba_m = mask_frequent(Aa, ka), mask_frequent(Ba, ka)
        Ac_m = normalize(sp.hstack([An_m, Aa_m], format="csr"))
        Bc_m = normalize(sp.hstack([Bn_m, Ba_m], format="csr"))
        rows, cols = [], []
        for A_, B_, k in [(An_m, Bn_m, TOPK["name"]), (Aa_m, Ba_m, TOPK["addr"]), (Ac_m, Bc_m, TOPK["comb"])]:
            r, cc = topk(A_, B_, k); rows.append(r); cols.append(cc)
        del An_m, Bn_m, Aa_m, Ba_m, Ac_m, Bc_m
        key = np.unique(np.concatenate(rows) * len(gb) + np.concatenate(cols))
        ra, rb = key // len(gb), key % len(gb)
        cn = pair_dot(An, Bn, ra, rb)
        ca = pair_dot(Aa, Ba, ra, rb)
        frames.append(pd.DataFrame({"ia": ga[ra].astype(np.int32), "ib": gb[rb].astype(np.int32),
                                    "cos_name": cn, "cos_addr": ca,
                                    "cos_comb": (cn + ca) / 2}))
        del An, Bn, Aa, Ba; gc.collect()
        log(f"  {c}: S1={len(ga):,} S23={len(gb):,} candidate pairs={len(key):,}")
    P = pd.concat(frames, ignore_index=True)
    return P


def group_stats(g, v):
    """Per-group rank (0 = best) of v (descending), group max of v and group size."""
    order = np.lexsort((-v, g))
    gs = g[order]
    starts = np.r_[0, np.flatnonzero(gs[1:] != gs[:-1]) + 1]
    counts = np.diff(np.r_[starts, len(gs)])
    rank = np.empty(len(g), np.int32); rank[order] = np.arange(len(gs)) - np.repeat(starts, counts)
    vmax = np.empty(len(g), np.float32); vmax[order] = np.repeat(v[order][starts], counts)
    cnt = np.empty(len(g), np.int32); cnt[order] = np.repeat(counts, counts)
    return rank, vmax, cnt


def add_context(P):
    ia, ib = P["ia"].values, P["ib"].values
    for col in ["cos_name", "cos_addr", "cos_comb"]:
        v = P[col].values
        r, m, n = group_stats(ia, v)
        P[f"a_rank_{col}"] = r; P[f"a_gap_{col}"] = v - m
        r, m, n2 = group_stats(ib, v)
        P[f"b_rank_{col}"] = r; P[f"b_gap_{col}"] = v - m
    P["a_n_cand"] = n
    P["b_n_cand"] = n2
    return P

## 4. Pair features

In [6]:
def _take(S, col, idx):
    return S[col].array.take(idx).tolist()


def pair_features(P, S1, S23, bs=PAIR_CHUNK):
    out = []
    for s in range(0, len(P), bs):
        ia, ib = P["ia"].values[s:s + bs], P["ib"].values[s:s + bs]
        a_core, b_core = _take(S1, "name_core", ia), _take(S23, "name_core", ib)
        a_full, b_full = _take(S1, "name_full", ia), _take(S23, "name_full", ib)
        b_af, b_al, b_tl = _take(S23, "alias_first", ib), _take(S23, "alias_last", ib), _take(S23, "name_tl", ib)
        a_addr, b_addr = _take(S1, "addr", ia), _take(S23, "addr", ib)
        a_nums, b_nums = _take(S1, "addr_nums", ia), _take(S23, "addr_nums", ib)
        a_hn, b_hn = _take(S1, "addr_hn", ia), _take(S23, "addr_hn", ib)
        a_ns = [x.replace(" ", "") for x in a_core]
        b_ns = [x.replace(" ", "") for x in b_core]

        def sim(x, y, scorer):
            return cpdist(x, y, scorer=scorer, workers=-1, dtype=np.float32)

        F = {
            "n_ratio": sim(a_core, b_core, fuzz.ratio),
            "n_tsort": sim(a_core, b_core, fuzz.token_sort_ratio),
            "n_tset": sim(a_core, b_core, fuzz.token_set_ratio),
            "n_partial": sim(a_core, b_core, fuzz.partial_ratio),
            "n_jw": sim(a_core, b_core, JaroWinkler.normalized_similarity),
            "nf_ratio": sim(a_full, b_full, fuzz.ratio),
            "nf_tset": sim(a_full, b_full, fuzz.token_set_ratio),
            "ns_ratio": sim(a_ns, b_ns, fuzz.ratio),
            "ns_partial": sim(a_ns, b_ns, fuzz.partial_ratio),
            "alias_first_ratio": sim(a_core, b_af, fuzz.token_sort_ratio),
            "alias_last_ratio": sim(a_core, b_al, fuzz.token_sort_ratio),
            "tl_ratio": sim(a_core, b_tl, fuzz.ratio),
            "tl_tset": sim(a_core, b_tl, fuzz.token_set_ratio),
            "ad_ratio": sim(a_addr, b_addr, fuzz.ratio),
            "ad_tsort": sim(a_addr, b_addr, fuzz.token_sort_ratio),
            "ad_tset": sim(a_addr, b_addr, fuzz.token_set_ratio),
            "ad_partial": sim(a_addr, b_addr, fuzz.partial_ratio),
            "num_tset": sim(a_nums, b_nums, fuzz.token_set_ratio),
            "num_ratio": sim(a_nums, b_nums, fuzz.ratio),
            "hn_ratio": sim(a_hn, b_hn, fuzz.ratio),
        }
        F["hn_in_b"] = np.fromiter((1 if h and (" " + h + " ") in (" " + n + " ") else 0
                                    for h, n in zip(a_hn, b_nums)), np.int8, len(ia))
        F["a_len_name"] = np.fromiter(map(len, a_core), np.int16, len(ia))
        F["b_len_name"] = np.fromiter(map(len, b_core), np.int16, len(ia))
        F["b_len_tl"] = np.fromiter(map(len, b_tl), np.int16, len(ia))
        F["a_len_addr"] = np.fromiter(map(len, a_addr), np.int16, len(ia))
        F["b_len_addr"] = np.fromiter(map(len, b_addr), np.int16, len(ia))
        F["a_n_nums"] = np.fromiter((len(x.split()) for x in a_nums), np.int8, len(ia))
        F["b_n_nums"] = np.fromiter((len(x.split()) for x in b_nums), np.int8, len(ia))
        F["b_nonascii"] = S23["name_nonascii"].values[ib]
        F["b_domain"] = S23["name_domain"].values[ib]
        F["b_is_s3"] = S23["is_s3"].values[ib]
        X = pd.DataFrame(F)
        ctx = P.iloc[s:s + bs].drop(columns=[c for c in ("ia", "ib", "y") if c in P.columns])
        X = pd.concat([X, ctx.reset_index(drop=True)], axis=1).astype(np.float32)
        out.append(X)
    return pd.concat(out, ignore_index=True)

## 5. Training data: normalise, block, label

In [7]:
S1, S23 = load_split("train")
P = block_split(S1, S23)
P = add_context(P)
log(f"train candidate pairs: {len(P):,} ({len(P) / len(S1):.1f} per S1 entity)")

gt = pd.read_csv(f"{DATA_DIR}/train/train_ground_truth.tsv", **READ_KW)
s1_pos = pd.Series(np.arange(len(S1)), index=S1["entity_id"].values)
g = gt[gt["matched_entity_ids"] != ""].assign(m=lambda d: d["matched_entity_ids"].str.split(",")).explode("m")
g = g[g["m"] != ""]
b_true = pd.Series(s1_pos.reindex(g["source1_entity_id"].values).values, index=g["m"].values)
b_true = b_true[~b_true.index.duplicated()]
true_ia = b_true.reindex(S23["entity_id"].values).fillna(-1).astype(np.int64).values
n_true_a = np.bincount(true_ia[true_ia >= 0], minlength=len(S1))
P["y"] = (true_ia[P["ib"].values] == P["ia"].values).astype(np.int8)
del gt, g, b_true; gc.collect()

n_true_pairs = int((true_ia >= 0).sum())
print(f"blocking recall (pairs): {P['y'].sum() / max(n_true_pairs, 1):.4f}   "
      f"reduction ratio: {1 - len(P) / (len(S1) * len(S23)):.6f}")

[     40s] train: read S1=2,206,821 S2+S3=10,320,219


ImportError: pyarrow>=13.0.0 is required for PyArrow backed StringArray.

## 6. Train / validation split (by Source 1 entity) and features

In [ ]:
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(S1))
n_val = min(N_VAL_S1, len(S1) // 5)
val_a = perm[:n_val]
trn_a = perm[n_val:n_val + N_TRN_S1]
role = np.zeros(len(S1), np.int8); role[trn_a] = 1; role[val_a] = 2

Pt = P[role[P["ia"].values] == 1].reset_index(drop=True)
Pv = P[role[P["ia"].values] == 2].reset_index(drop=True)
del P; gc.collect()
log(f"train pairs {len(Pt):,} (pos {Pt['y'].mean():.3f}) | val pairs {len(Pv):,}")

Xt = pair_features(Pt, S1, S23); log("train features done")
Xv = pair_features(Pv, S1, S23); log("val features done")
FEATURES = list(Xt.columns)
print(len(FEATURES), "features")

## 7. LightGBM matcher

In [ ]:
params = dict(objective="binary", learning_rate=0.05, num_leaves=127, min_data_in_leaf=100,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              num_threads=N_JOBS, verbose=-1, seed=SEED)
dtr = lgb.Dataset(Xt, Pt["y"].values, free_raw_data=True)
dva = lgb.Dataset(Xv, Pv["y"].values, reference=dtr)
model = lgb.train(params, dtr, num_boost_round=3000, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)])
model.save_model(f"{MODEL_DIR}/lgb_matcher.txt")
pv = model.predict(Xv, num_threads=N_JOBS).astype(np.float32)

imp = pd.Series(model.feature_importance("gain"), index=FEATURES).sort_values(ascending=False)
imp.head(25)

## 8. Decision rule tuned for macro F0.5

* keep pairs with probability `>= t`
* one-to-one: each S2/S3 record goes to its single best S1 entity
* optionally drop matches whose probability is below `r x` the entity's best match probability

F0.5 per entity simplifies to `1.25 * TP / (n_pred + 0.25 * n_true)` (1.0 if both are empty).
Note: in validation only the held-out S1 entities compete for a record, so the one-to-one step
is slightly weaker here than on the full test set.

In [ ]:
def select_matches(ia, ib, p, t, r=0.0):
    m = p >= t
    d = pd.DataFrame({"ia": ia[m], "ib": ib[m], "p": p[m]})
    d = d.sort_values("p", ascending=False, kind="stable").drop_duplicates("ib")
    if r > 0 and len(d):
        best = d.groupby("ia")["p"].transform("max")
        d = d[d["p"] >= r * best]
    return d


def macro_f05(sel_ia, sel_tp, eval_a, n_true_a):
    n = len(n_true_a)
    npred = np.bincount(sel_ia, minlength=n)[eval_a]
    tp = np.bincount(sel_ia, weights=sel_tp, minlength=n)[eval_a]
    nt = n_true_a[eval_a]
    f = np.where((npred == 0) & (nt == 0), 1.0, 1.25 * tp / np.maximum(npred + 0.25 * nt, 1e-9))
    return f.mean()


ia_v, ib_v, y_v = Pv["ia"].values, Pv["ib"].values, Pv["y"].values
results = []
for t in np.arange(0.20, 0.96, 0.05):
    for r in [0.0, 0.3, 0.5]:
        d = select_matches(ia_v, ib_v, pv, t, r)
        tp = (true_ia[d["ib"].values] == d["ia"].values).astype(float)
        results.append((round(t, 2), r, macro_f05(d["ia"].values, tp, val_a, n_true_a)))
res = pd.DataFrame(results, columns=["t", "r", "F05"]).sort_values("F05", ascending=False)
BEST_T, BEST_R = float(res.iloc[0]["t"]), float(res.iloc[0]["r"])

upper = macro_f05(ia_v[y_v == 1], np.ones((y_v == 1).sum()), val_a, n_true_a)
print(f"best threshold t={BEST_T}, r={BEST_R}  ->  validation macro F0.5 = {res.iloc[0]['F05']:.4f}")
print(f"(upper bound given blocking = {upper:.4f})")
json.dump({"threshold": BEST_T, "relative": BEST_R, "features": FEATURES},
          open(f"{MODEL_DIR}/decision.json", "w"))
res.head(10)

In [ ]:
# free training memory before the test phase
del S1, S23, Pt, Pv, Xt, Xv, dtr, dva, pv, true_ia, n_true_a, role
gc.collect()

## 9. Test inference

In [ ]:
S1, S23 = load_split("test")
P = block_split(S1, S23)
P = add_context(P)
log(f"test candidate pairs: {len(P):,} ({len(P) / len(S1):.1f} per S1 entity)")

p_test = np.empty(len(P), np.float32)
CH = 2_000_000
for s in range(0, len(P), CH):
    X = pair_features(P.iloc[s:s + CH].reset_index(drop=True), S1, S23)
    p_test[s:s + CH] = model.predict(X[FEATURES], num_threads=N_JOBS)
    log(f"  scored {min(s + CH, len(P)):,}/{len(P):,}")
del X; gc.collect()

sel = select_matches(P["ia"].values, P["ib"].values, p_test, BEST_T, BEST_R)
log(f"selected matches: {len(sel):,};  S1 entities with >=1 match: {sel['ia'].nunique() / len(S1):.3f}")

## 10. Write submission files and validate

In [ ]:
def write_id_lists(path, header, S1, S23, ia, ib):
    ids = S23["entity_id"].values[ib]
    d = pd.DataFrame({"ia": ia, "id": ids})
    lists = d.groupby("ia", sort=False)["id"].agg(",".join)
    col = np.full(len(S1), "", dtype=object)
    col[lists.index.values] = lists.values
    pd.DataFrame({header[0]: S1["entity_id"].values, header[1]: col}).to_csv(
        path, sep="\t", index=False, quoting=csv.QUOTE_NONE)


write_id_lists(f"{OUT_DIR}/matching_results.tsv", ["source1_entity_id", "matched_entity_ids"],
               S1, S23, sel["ia"].values, sel["ib"].values)
write_id_lists(f"{OUT_DIR}/candidate_pairs.tsv", ["source1_entity_id", "candidate_entity_ids"],
               S1, S23, P["ia"].values, P["ib"].values)
log("files written")

r = subprocess.run([sys.executable, "utils/validate_submission.py",
                    "--matching", f"{OUT_DIR}/matching_results.tsv",
                    "--candidate", f"{OUT_DIR}/candidate_pairs.tsv",
                    "--test-dir", f"{DATA_DIR}/test"], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:])

In [ ]:
pd.read_csv(f"{OUT_DIR}/matching_results.tsv", sep="\t", keep_default_na=False, nrows=10)